# Part III · Lesson 3 — Search by meaning and by exact words

## Import the document and model helpers

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

## Connect the two kinds of model

In [ ]:
load_dotenv()
# Groq generates answers; OpenAI creates search embeddings.
model = init_chat_model(
    "openai/gpt-oss-120b", model_provider="groq", reasoning_effort="low",
)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## Find the supplied text documents

In [ ]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Run this notebook with its own directory as the working directory.
data_dir = Path("../doc/products")
text_files = sorted(data_dir.glob("*.txt"))
print("Files:", len(text_files))

## Keep text and source together

In [ ]:
documents = [
    Document(
        page_content=path.read_text(encoding="utf-8"),
        metadata={"source": path.name},
    )
    for path in text_files
]
print(documents[0].metadata)

## Split the documents into searchable pieces

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks = splitter.split_documents(documents)
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"{chunk.metadata['source']}:{index}"
print("Documents:", len(documents), "Chunks:", len(chunks))

## Build the vector index

In [ ]:
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

## Prepare readable context with source labels

In [ ]:
def format_context(documents: list[Document]) -> str:
    return "\n\n".join(
        f"[{doc.metadata['source']}]\n{doc.page_content}"
        for doc in documents
    )

## Use the same tokenizer for documents and questions

In [ ]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list[str]:
    return re.findall(r"\b[\w-]+\b", text.lower())

bm25 = BM25Okapi([tokenize(chunk.page_content) for chunk in chunks])

## Return only keyword hits with a positive score

In [ ]:
def keyword_search(question: str, k: int = 5) -> list[Document]:
    scores = bm25.get_scores(tokenize(question))
    ranked = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)
    return [chunks[index] for index in ranked[:k] if scores[index] > 0]

## Inspect both methods for one error code

In [ ]:
question = "What does E922 mean?"
vector_results = vector_store.similarity_search(question, k=5)
keyword_results = keyword_search(question)
print("Vector IDs:", [doc.metadata["chunk_id"] for doc in vector_results])
print("Keyword IDs:", [doc.metadata["chunk_id"] for doc in keyword_results])

## Combine rankings without duplicating a chunk

In [ ]:
def reciprocal_rank_fusion(vector_results, keyword_results, final_k=3):
    scores, documents_by_id = {}, {}
    for results in [vector_results, keyword_results]:
        for rank, document in enumerate(results, start=1):
            chunk_id = document.metadata["chunk_id"]
            documents_by_id[chunk_id] = document
            scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (60 + rank)
    ranked_ids = sorted(scores, key=scores.get, reverse=True)
    return [documents_by_id[chunk_id] for chunk_id in ranked_ids[:final_k]]

## Make hybrid retrieval reusable

In [ ]:
def hybrid_search(question: str) -> list[Document]:
    vector_results = vector_store.similarity_search(question, k=5)
    keyword_results = keyword_search(question, k=5)
    return reciprocal_rank_fusion(vector_results, keyword_results)

hybrid_results = hybrid_search("What does E922 mean?")
for document in hybrid_results:
    print(document.metadata["source"], document.page_content[:120])

## Generate an answer and keep its evidence

In [ ]:
def ask_hybrid_rag(question: str) -> dict:
    retrieved_documents = hybrid_search(question)
    instruction = (
        "Answer using only the context. If it is insufficient, say you could not find the answer. "
        "Cite filenames in brackets.\nContext:\n" + format_context(retrieved_documents)
    )
    answer = model.invoke([SystemMessage(instruction), HumanMessage(question)]).text
    return {
        "question": question, "answer": answer, "documents": retrieved_documents,
        "sources": sorted({doc.metadata["source"] for doc in retrieved_documents}),
    }

## Try an exact code and a natural description

In [ ]:
code_result = ask_hybrid_rag("What does E922 mean and how do I fix it?")
description_result = ask_hybrid_rag("The brush on my robot vacuum is jammed. What should I do?")
print("Code:", code_result["answer"], code_result["sources"])
print("Description:", description_result["answer"], description_result["sources"])